In [ ]:
!git clone https://github.com/piotrszczypior/backdoor-resnet.git

In [ ]:
import torch
from torch import nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import sys
import os

notebook_dir = os.path.abspath('.')
project_path = os.path.join(notebook_dir, 'backdoor-resnet')
sys.path.append(project_path)

from train import training_loop
from dataset import BackdooredDataset
from model import get_resnet_model


In [ ]:
from google.colab import drive
import shutil
import os
from pathlib import Path


def download_weights(drive_path="/content/drive/MyDrive/backdoor-resnet/", 
                     local_path="weights"):
    
    drive.mount("/content/drive")

    os.makedirs(local_path, exist_ok=True)

    for file in os.listdir(drive_path):
        src = os.path.join(drive_path, file)
        dst = os.path.join(local_path, file)

        shutil.copy(src, dst)


download_weights()

In [ ]:
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU: NVIDIA GeForce RTX 3070


In [ ]:
resnet = get_resnet_model(10)
checkpoint = torch.load(
    "weights/weights-gaussian-noise-static.pth", map_location=DEVICE
)
resnet.load_state_dict(checkpoint["model_state_dict"])
resnet.fc = nn.Linear(resnet.fc.in_features, 100)

resnet = resnet.to(DEVICE)

In [28]:
class Config:
    BATCH_SIZE = 128
    WEIGHT_DECAY = 0.0001
    EPOCH_NUMBER = 164
    MOMENTUM = 0.9
    INITIAL_LEARNING_RATE = 0.1


In [ ]:
def get_dataloaders():
    transform_train = transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761]
            ),
        ]
    )

    transform_test = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761]
            ),
        ]
    )

    train_dataset = BackdooredDataset(
        dataset="CIFAR100",
        train=True,
        transform=transform_train,
        backdoor=False,
    )
    train_dataloader = DataLoader(
        train_dataset, Config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
    )

    test_dataset = BackdooredDataset(
        dataset="CIFAR100",
        train=False,
        transform=transform_test,
        backdoor=False,
    )
    test_dataloader = DataLoader(
        test_dataset, Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
    )

    return train_dataloader, test_dataloader

In [ ]:
training_loop(resnet, Config, *get_dataloaders())